# 08 — Telephony — Plivo

V3 webhook signature (HMAC-SHA256), Plivo XML answer, AMD, native DTMF send (the parity gain over Twilio), ring timeout, status callback, voicemail drop. Plivo streams mulaw 8 kHz over the same WebSocket-media model as Twilio — we pin the codec in the answer XML.


## Optional: run in Docker

Uncomment the cell below to launch JupyterLab + EmbeddedServer in a container. Container helpers for the TypeScript notebooks are tracked separately — see issue #80 for status.


In [ ]:
// Optional — launch the patter-notebooks Docker stack from this cell.
// Skip this cell to run on your host runtime. Idempotent if uncommented.
//
// import * as _setup from "./_setup.ts";
// await _setup.startDocker();
// await _setup.startDocker({ openUrl: true });


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | _none — offline crypto demos_ |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + Plivo creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Plivo carrier.


In [ ]:
import { Patter, Plivo } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Plivo({
      authId: 'MA-test-only',
      authToken: 'test',  // gitleaks:allow
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Plivo, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Plivo({ authId: 'MA-test-only', authToken: 'test' }),  // gitleaks:allow
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


## §2: Feature Tour

Exercises Plivo carrier construction, V3 webhook signature verification, the Plivo answer XML, and the AMD classifier. All offline.


### V3 signature — POST with form params (the real production case)

Plivo's V3 scheme: `signed = url + sorted_post_params + "." + nonce` for POST (params sorted alphabetically by key, concatenated as `key1value1…`), or `url + "." + nonce` for GET. HMAC-SHA256 keyed on `authToken`, base64-encoded.


In [ ]:
import { validatePlivoSignature } from "getpatter/telephony/plivo";
import { createHmac } from "node:crypto";
await cell('plivo_v3_sig_post', { tier: 1, env }, () => {
  const token = 'test_plivo_auth_token_32chars___';  // gitleaks:allow
  const url   = 'https://example.com/webhooks/plivo/voice';
  const nonce = 'b3a1f0c4-2025-4f6e-9a52-7e34a8b8e0c2';
  const params = { CallUUID: 'CU_xyz', From: '+15551112222', To: '+15553334444' };
  const sortedConcat = Object.keys(params).sort().map(k => `${k}${(params as any)[k]}`).join('');
  const signed = `${url}${sortedConcat}.${nonce}`;
  const sig = createHmac('sha256', token).update(signed).digest('base64');
  const valid = validatePlivoSignature(url, nonce, sig, token, params, 'POST');
  console.log(`signed prefix: ${signed.slice(0, 80)}...`);
  console.log(`signature    : ${sig}`);
  console.log(`valid        : ${valid}`);
  if (!valid) throw new Error('POST signature should be valid');
});


### V3 signature — GET (e.g. the `/webhooks/plivo/transfer` aleg_url)

Query params live in the URL already, so the signed string is just `url + "." + nonce`.


In [ ]:
import { validatePlivoSignature } from "getpatter/telephony/plivo";
import { createHmac } from "node:crypto";
await cell('plivo_v3_sig_get', { tier: 1, env }, () => {
  const token = 'test_plivo_auth_token_32chars___';  // gitleaks:allow
  const url   = 'https://example.com/webhooks/plivo/transfer?to=%2B15551234567';
  const nonce = 'b3a1f0c4-2025-4f6e-9a52-7e34a8b8e0c2';
  const sig = createHmac('sha256', token).update(`${url}.${nonce}`).digest('base64');
  const valid = validatePlivoSignature(url, nonce, sig, token, undefined, 'GET');
  console.log(`valid: ${valid}`);
  if (!valid) throw new Error('GET signature should be valid');
});


### V3 signature — tampered request rejected

Tampering with any POST param value invalidates the signature.


In [ ]:
import { validatePlivoSignature } from "getpatter/telephony/plivo";
import { createHmac } from "node:crypto";
await cell('plivo_v3_sig_tampered', { tier: 1, env }, () => {
  const token = 'test_plivo_auth_token_32chars___';  // gitleaks:allow
  const url = 'https://example.com/webhooks/plivo/voice', nonce = 'n';
  const original = { CallUUID: 'CU1', From: '+15551112222' };
  const concat = Object.keys(original).sort().map(k => `${k}${(original as any)[k]}`).join('');
  const sig = createHmac('sha256', token).update(`${url}${concat}.${nonce}`).digest('base64');
  const tampered = { CallUUID: 'CU1', From: '+15559999999' };
  const valid = validatePlivoSignature(url, nonce, sig, token, tampered, 'POST');
  console.log(`tampered valid: ${valid}  (expected false)`);
  if (valid) throw new Error('tampered signature must be rejected');
});


### Plivo answer XML

Unlike Twilio, Plivo embeds the WSS URL as the `<Stream>` element's **text content** (not a `url=` attribute). The `&` between query parameters MUST be escaped to `&amp;`.


In [ ]:
import { PlivoAdapter } from "getpatter";
await cell('plivo_answer_xml', { tier: 1, env }, () => {
  const xml = PlivoAdapter.generateStreamXml(
    'wss://example.com/ws/plivo/stream/CALLUUID?caller=%2B1&callee=%2B2',
    'audio/x-mulaw;rate=8000',
    { 'X-PH-caller': '+1', 'X-PH-callee': '+2' },
  );
  console.log(xml);
  if (!xml.includes('<Stream') || !xml.includes('bidirectional="true"'))
    throw new Error('missing <Stream> with bidirectional');
  if (!xml.includes('&amp;callee'))
    throw new Error('& must be XML-escaped');
});


### AMD classifier — defensive across Plivo result spellings

Plivo's async machine-detection callback reports the outcome via a result field whose spelling varies by API version. `classifyPlivoAmd` (in `getpatter`'s Plivo telephony module) normalises common shapes to `human` / `machine` / `fax` / `unknown`.

### Native DTMF send — a parity *gain* over Twilio

Plivo accepts a `sendDTMF` command over the media WebSocket. The `PlivoBridge.sendDtmf` method takes the WS + digits and writes a frame after filtering invalid characters.


In [ ]:
import { PlivoBridge } from "getpatter";
await cell('plivo_send_dtmf', { tier: 1, env }, async () => {
  const sent: any[] = [];
  const fakeWs = { send: (t: string) => sent.push(JSON.parse(t)) };
  const bridge = new PlivoBridge({
    telephonyProvider: 'plivo',
    plivoAuthId: 'MA-test-only',
    plivoAuthToken: 'tok',
    phoneNumber: '+15555550100',
    webhookUrl: 'example.com',
  });
  await bridge.sendDtmf(fakeWs as any, 'callid', '12ab#xZ', 0);
  console.log(sent[0]);
  if (JSON.stringify(sent[0]) !== JSON.stringify({ event: 'sendDTMF', dtmf: '12ab#' }))
    throw new Error('unexpected DTMF frame');
});


### E.164 phone number patterns


In [ ]:
await cell('e164_patterns', { tier: 1, env }, () => {
  const E164 = /^\+[1-9]\d{6,14}$/;
  const cases: Array<[string, boolean]> = [
    ['+15555550100', true],
    ['+442071838750', true],
    ['+393399123456', true],
    ['15555550100',  false],
    ['+1',          false],
    ['not-a-number', false],
  ];
  for (const [n, expected] of cases) {
    const result = E164.test(n);
    const mark = result === expected ? '✓' : '✗';
    console.log(`  ${mark} ${JSON.stringify(n).padEnd(20)} → ${result}`);
    if (result !== expected) throw new Error(`unexpected ${n}`);
  }
});


### Plivo carrier construction


In [ ]:
import { Patter, Plivo } from "getpatter";
await cell('plivo_carrier', { tier: 1, env }, () => {
  const carrier = new Plivo({
    authId: 'MA-test-only',
    authToken: 'test_token',  // gitleaks:allow
  });
  const p = new Patter({
    carrier,
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log(`carrier kind: ${carrier.kind}`);
  console.log(`phone:        +15555550100`);
  if (carrier.kind !== 'plivo') throw new Error(`expected plivo, got ${carrier.kind}`);
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.

Requires `PLIVO_AUTH_ID`, `PLIVO_AUTH_TOKEN`, `PLIVO_PHONE_NUMBER`, `TARGET_PHONE_NUMBER` and a public webhook URL.


### Pre-flight checklist


In [ ]:
await cell(
  'live_preflight',
  { tier: 4, env, required: ['PLIVO_AUTH_ID', 'PLIVO_AUTH_TOKEN', 'PLIVO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER'] },
  () => {
    console.log(`  carrier:  Plivo ${env.plivoNumber}  →  ${env.targetNumber}`);
    console.log(`  webhook:  ${env.publicWebhookUrl || '(ngrok auto-launch)'}`);
    console.log(`  features: V3 signature + AMD + voicemail fallback`);
  },
);


### Live Plivo call with AMD *(T4)*


In [ ]:
import { Patter, Plivo, OpenAIRealtime } from "getpatter";
await cell(
  'live_plivo_amd',
  { tier: 4, env, required: ['PLIVO_AUTH_ID', 'PLIVO_AUTH_TOKEN', 'PLIVO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'] },
  async () => {
    const p = new Patter({
      carrier: new Plivo({ authId: env.plivoAuthId, authToken: env.plivoAuthToken }),
      phoneNumber: env.plivoNumber,
      webhookUrl: env.publicWebhookUrl,
    });
    const agent = p.agent({
      systemPrompt: 'You are a Plivo telephony demo agent. Greet the caller and hang up.',
      engine: new OpenAIRealtime({ apiKey: env.openaiKey }),
    });
    await p.call({
      to: env.targetNumber,
      agent,
      machineDetection: true,
      voicemailMessage: 'You reached Patter demo. Goodbye.',
      firstMessage: 'Hello from Patter Plivo demo.',
      ringTimeout: env.maxCallSeconds,
    });
    console.log('✓ Plivo AMD call completed');
  },
);
